In [1]:
# ===========================================
# 금융상품 추천 시스템 개발(XGBoost)
# ===========================================

# 필요한 모든 라이브러리 import
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

# 머신러닝 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score

# XGBoost
try:
    from xgboost import XGBClassifier
    import xgboost as xgb
    print("✅ 모든 라이브러리 import 성공!")
except ImportError as e:
    print(f"❌ Import 오류: {e}")
    print("XGBoost 설치: pip install xgboost")

print("🚀 XGBoost 금융상품 추천 시스템")
print("=" * 50)

✅ 모든 라이브러리 import 성공!
🚀 XGBoost 금융상품 추천 시스템


In [2]:
target_names = {
    'LIQ': '유동성자산',
    'CDS': '양도성예금증서', 
    'NMMF': '비머니마켓펀드',
    'STOCKS': '주식보유',
    'RETQLIQ': '퇴직준비금유동성'
}

In [3]:
# 데이터 로드 시도
try:
    df = pd.read_csv('data/cleaned_scf_data.csv')
    X = df.iloc[:,:-5]  # 14개 독립변수
    y = df.iloc[:,-5:]  # 5개 타겟변수
    print(f"✅ 실제 데이터 로드: {X.shape[0]:,}명, {X.shape[1]}개 특성")
except FileNotFoundError:
    print("※ 실제 데이터 없음. 샘플 데이터 생성 중...")
    
    # 샘플 데이터 생성
    np.random.seed(42)
    n_samples = 2000
    X = pd.DataFrame(np.random.randn(n_samples, 16), 
                    columns=[f'feature_{i+1}' for i in range(16)])
    y = pd.DataFrame(np.random.randint(0, 2, (n_samples, 5)),
                    columns=list(target_names.keys()))
    print(f"✅ 샘플 데이터 생성: {X.shape[0]:,}명, {X.shape[1]}개 특성")

print(f"데이터 준비 완료: X={X.shape}, y={y.shape}")

✅ 실제 데이터 로드: 22,975명, 14개 특성
데이터 준비 완료: X=(22975, 14), y=(22975, 5)


In [4]:
# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"훈련 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")
print("✅ 데이터 분할 완료")

훈련 데이터: (18380, 14)
테스트 데이터: (4595, 14)
✅ 데이터 분할 완료


In [32]:
# 셀 7: 팀원 설정에 맞춘 XGBoost 모델 생성
import numpy as np

# 특성 개수 확인 (colsample_bytree 계산용)
n_features = X.shape[1] 
colsample_ratio = np.sqrt(n_features) / n_features  # sqrt(16)/16 = 0.25

# 팀원 RandomForest 설정을 XGBoost로 변환
base_xgb_model = XGBClassifier(
    # === 직접 매핑 ===
    n_estimators=500,           # 팀원과 동일
    max_depth=20,              # 팀원과 동일  
    random_state=42,           # 팀원과 동일
    n_jobs=-1,                 # 팀원과 동일
    
    # === 유사 효과 매핑 ===
    min_child_weight=5,        # min_samples_leaf=5와 유사
    colsample_bytree=0.25,     # max_features='sqrt'와 유사
    
    # === RandomForest의 min_samples_split 대체 ===
    gamma=0.1,                 # 분할 최소 이득 (과적합 방지)
    reg_alpha=0.1,             # L1 정규화
    reg_lambda=1.0,            # L2 정규화
    
    # === XGBoost 고유 설정 ===
    learning_rate=0.1,         # 학습률
    subsample=0.8,             # RandomForest bootstrap과 유사
    
    # === 기타 ===
    eval_metric='logloss',
    verbosity=0
)

# class_weight='balanced' 처리를 위한 MultiOutputClassifier
multilabel_xgb = MultiOutputClassifier(base_xgb_model)

print("✅ 팀원 RandomForest 설정에 맞춘 XGBoost 모델 생성 완료")
print(f"📊 특성 개수: {n_features}, colsample_bytree: {colsample_ratio:.3f}")

✅ 팀원 RandomForest 설정에 맞춘 XGBoost 모델 생성 완료
📊 특성 개수: 14, colsample_bytree: 0.267


In [18]:
# multilabel_xgb 모델 훈련
print("🚀 MultiOutputClassifier 훈련 시작...")

# 데이터 확인
if 'X_train' in locals() and 'y_train' in locals():
    print(f"훈련 데이터: {X_train.shape}, 타겟: {y_train.shape}")
    
    # 훈련 실행
    multilabel_xgb.fit(X_train, y_train)
    print("✅ MultiOutputClassifier 훈련 완료!")
    
    # 테스트 예측
    test_pred = multilabel_xgb.predict(X_test)
    print(f"예측 결과 형태: {test_pred.shape}")
    
else:
    print("❌ 훈련 데이터가 없습니다.")
    print("먼저 데이터 로드 및 분할을 실행해주세요.")

🚀 MultiOutputClassifier 훈련 시작...
훈련 데이터: (18380, 14), 타겟: (18380, 5)
✅ MultiOutputClassifier 훈련 완료!
예측 결과 형태: (4595, 5)


In [6]:
print("🚀 XGBoost 모델 훈련 시작...")
import time

start_time = time.time()
multilabel_xgb.fit(X_train, y_train)
train_time = time.time() - start_time

print(f"✅ XGBoost 모델 훈련 완료! (소요시간: {train_time:.1f}초)")

🚀 XGBoost 모델 훈련 시작...
✅ XGBoost 모델 훈련 완료! (소요시간: 1.8초)


In [7]:
# 테스트 데이터 예측
print("🔍 모델 예측 중...")
test_pred = multilabel_xgb.predict(X_test)
print("✅ 예측 완료")

🔍 모델 예측 중...
✅ 예측 완료


In [8]:
# 전체 성능 지표
accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, average='samples', zero_division=0)
recall = recall_score(y_test, test_pred, average='samples', zero_division=0)
f1 = f1_score(y_test, test_pred, average='samples', zero_division=0)

print("★ XGBoost 전체 모델 성능")
print("=" * 40)
print(f"정확도 (Accuracy):  {accuracy:.3f}")
print(f"정밀도 (Precision): {precision:.3f}")
print(f"재현율 (Recall):    {recall:.3f}")
print(f"F1 점수:           {f1:.3f}")

★ XGBoost 전체 모델 성능
정확도 (Accuracy):  0.830
정밀도 (Precision): 0.963
재현율 (Recall):    0.942
F1 점수:           0.944


In [9]:
print("🎯 포트폴리오 밸런싱 & 시나리오 기반 추천 시스템")
print("=" * 60)

# =====================================
# 1. 포트폴리오 밸런싱 시스템
# =====================================

class PortfolioBalancer:
    """개인 맞춤형 포트폴리오 균형 분석 및 추천"""
    
    def __init__(self):
        # 연령별 권장 자산 배분 (일반적인 금융업계 가이드라인)
        self.age_based_allocation = {
            'young': {'age_range': (20, 35), 'safe_ratio': 0.3, 'growth_ratio': 0.7},    # 젊은층: 성장 70%
            'middle': {'age_range': (36, 50), 'safe_ratio': 0.5, 'growth_ratio': 0.5},   # 중년층: 균형 50:50
            'mature': {'age_range': (51, 65), 'safe_ratio': 0.7, 'growth_ratio': 0.3},   # 장년층: 안전 70%
            'senior': {'age_range': (66, 100), 'safe_ratio': 0.9, 'growth_ratio': 0.1}   # 시니어: 안전 90%
        }
        
        # 상품별 자산 분류
        self.asset_classification = {
            'safe_assets': ['LIQ', 'CDS', 'RETQLIQ'],        # 안전자산: 예금, 퇴직준비금
            'growth_assets': ['STOCKS', 'NMMF']              # 성장자산: 주식, 펀드
        }
    
    def get_age_group(self, age):
        """나이에 따른 그룹 분류"""
        for group, info in self.age_based_allocation.items():
            if info['age_range'][0] <= age <= info['age_range'][1]:
                return group
        return 'middle'  # 기본값
    
    def analyze_current_portfolio(self, predictions, age):
        """현재 예측된 포트폴리오 분석"""
        age_group = self.get_age_group(age)
        recommended_allocation = self.age_based_allocation[age_group]
        
        # 현재 포트폴리오 계산
        safe_prob = sum([predictions.get(asset, 0) for asset in self.asset_classification['safe_assets']])
        growth_prob = sum([predictions.get(asset, 0) for asset in self.asset_classification['growth_assets']])
        total_prob = safe_prob + growth_prob
        
        if total_prob > 0:
            current_safe_ratio = safe_prob / total_prob
            current_growth_ratio = growth_prob / total_prob
        else:
            current_safe_ratio = current_growth_ratio = 0
        
        # 권장 비율과 비교
        safe_gap = recommended_allocation['safe_ratio'] - current_safe_ratio
        growth_gap = recommended_allocation['growth_ratio'] - current_growth_ratio
        
        return {
            'age_group': age_group,
            'current': {'safe': current_safe_ratio, 'growth': current_growth_ratio},
            'recommended': {'safe': recommended_allocation['safe_ratio'], 'growth': recommended_allocation['growth_ratio']},
            'gap': {'safe': safe_gap, 'growth': growth_gap}
        }
    
    def get_rebalancing_advice(self, portfolio_analysis, user_features):
        """포트폴리오 리밸런싱 조언"""
        age = user_features[4]
        income = user_features[5]
        risk_tolerance = user_features[6]
        
        advice = []
        analysis = portfolio_analysis
        
        # 연령대별 기본 메시지
        age_group_msg = {
            'young': f"🌟 {age}세 청년층",
            'middle': f"💼 {age}세 중년층", 
            'mature': f"🎯 {age}세 장년층",
            'senior': f"🏖️ {age}세 시니어층"
        }
        
        advice.append(f"\n📊 **포트폴리오 분석** - {age_group_msg[analysis['age_group']]}")
        advice.append(f"현재 구성: 안전자산 {analysis['current']['safe']:.1%}, 성장자산 {analysis['current']['growth']:.1%}")
        advice.append(f"권장 구성: 안전자산 {analysis['recommended']['safe']:.1%}, 성장자산 {analysis['recommended']['growth']:.1%}")
        
        # 리밸런싱 필요성 판단
        if abs(analysis['gap']['safe']) > 0.2:  # 20% 이상 차이
            if analysis['gap']['safe'] > 0:  # 안전자산 부족
                advice.append(f"\n🛡️ **안전자산 {analysis['gap']['safe']:.1%} 추가 권장**")
                advice.append("   → 예금, 적금 상품을 늘려 안정성을 확보하세요")
                recommendation = "더 많은 예금/적금 상품 추천"
            else:  # 성장자산 부족
                advice.append(f"\n📈 **성장자산 {abs(analysis['gap']['growth']):.1%} 추가 권장**")
                advice.append("   → 주식, 펀드 상품으로 수익성을 높이세요")
                recommendation = "더 많은 투자 상품 추천"
        else:
            advice.append(f"\n✅ **균형잡힌 포트폴리오**")
            advice.append("   → 현재 자산 배분이 연령대에 적합합니다")
            recommendation = "현재 구성 유지"
        
        # 소득 및 위험성향 고려
        if income < 3000:  # 저소득
            advice.append(f"\n💡 **소득 고려사항** ({income:,}만원)")
            advice.append("   → 안전자산 위주로 기초를 다진 후 점진적 투자 권장")
        elif income > 8000:  # 고소득
            advice.append(f"\n💰 **고소득자 전략** ({income:,}만원)")
            advice.append("   → 세제혜택 상품과 함께 적극적 자산 배분 가능")
        
        return {
            'advice': advice,
            'recommendation': recommendation,
            'priority_assets': 'safe' if analysis['gap']['safe'] > 0 else 'growth'
        }

🎯 포트폴리오 밸런싱 & 시나리오 기반 추천 시스템


In [10]:
# =====================================
# 2. 시나리오 기반 추천 시스템  
# =====================================

class ScenarioRecommendationSystem:
    """생애 이벤트별 맞춤 금융상품 추천"""
    
    def __init__(self):
        self.scenarios = {
            'marriage': {
                'name': '💒 결혼 준비',
                'description': '결혼식 및 신혼 생활 자금 마련',
                'typical_period': '2-3년',
                'target_amount_ratio': 1.5,  # 연소득 대비 배수
                'priority_products': ['bank_deposits', 'bank_savings'],
                'risk_level': 'low'
            },
            'children_education': {
                'name': '👶 자녀 교육비',
                'description': '자녀 교육비 및 대학 등록금 준비',
                'typical_period': '10-18년',
                'target_amount_ratio': 3.0,
                'priority_products': ['bank_savings', 'investment_funds'],
                'risk_level': 'medium'
            },
            'house_purchase': {
                'name': '🏠 내집 마련',
                'description': '주택 구입을 위한 자금 마련',
                'typical_period': '5-10년',
                'target_amount_ratio': 5.0,
                'priority_products': ['bank_savings', 'mortgage_loans'],
                'risk_level': 'medium'
            },
            'retirement': {
                'name': '🏖️ 은퇴 준비',
                'description': '노후 생활비 및 연금 준비',
                'typical_period': '20-30년',
                'target_amount_ratio': 10.0,
                'priority_products': ['pension_savings', 'investment_funds'],
                'risk_level': 'medium-high'
            },
            'emergency_fund': {
                'name': '🚨 비상 자금',
                'description': '갑작스런 상황 대비 비상 자금',
                'typical_period': '즉시-1년',
                'target_amount_ratio': 0.5,
                'priority_products': ['bank_deposits'],
                'risk_level': 'very_low'
            }
        }
    
    def recommend_scenario(self, user_features):
        """사용자 특성 기반 적합한 시나리오 추천"""
        age = user_features[4]
        income = user_features[5]
        marriage_status = user_features[12]
        children_count = user_features[13]
        
        scenario_scores = {}
        
        # 연령별 시나리오 적합도
        for scenario_id, scenario in self.scenarios.items():
            score = 0
            
            if scenario_id == 'marriage':
                score += 100 if (age >= 25 and age <= 35 and marriage_status == 0) else 20
                
            elif scenario_id == 'children_education':
                score += 100 if (marriage_status == 1 and children_count > 0) else 30
                score += min(children_count * 20, 60)
                
            elif scenario_id == 'house_purchase':
                score += 80 if (age >= 28 and age <= 45) else 40
                score += 20 if income >= 4000 else 0
                
            elif scenario_id == 'retirement':
                score += max(0, age - 30) * 2  # 나이가 많을수록 높은 점수
                score += 20 if income >= 6000 else 10
                
            elif scenario_id == 'emergency_fund':
                score += 60  # 모든 사람에게 기본적으로 필요
                score += 20 if income < 5000 else 10  # 저소득층에게 더 중요
            
            scenario_scores[scenario_id] = score
        
        # 상위 3개 시나리오 반환
        top_scenarios = sorted(scenario_scores.items(), key=lambda x: x[1], reverse=True)[:3]
        return top_scenarios
    
    def create_scenario_plan(self, scenario_id, user_features, target_years=None):
        """특정 시나리오에 대한 구체적 계획 수립"""
        if scenario_id not in self.scenarios:
            return None
            
        scenario = self.scenarios[scenario_id]
        age = user_features[4]
        income = user_features[5]
        assets = user_features[15]
        risk_tolerance = user_features[6]
        
        # 목표 금액 계산
        target_amount = income * scenario['target_amount_ratio'] * 10000  # 만원 단위
        
        # 목표 기간 설정
        if target_years is None:
            if scenario_id == 'marriage':
                target_years = max(2, min(5, (35 - age)))
            elif scenario_id == 'children_education':
                target_years = max(10, 18 - user_features[13] * 3) if user_features[13] > 0 else 15
            elif scenario_id == 'house_purchase':
                target_years = max(3, min(10, (40 - age) // 2))
            elif scenario_id == 'retirement':
                target_years = max(10, 65 - age)
            else:
                target_years = 3
        
        # 월 저축 필요 금액
        monthly_saving = target_amount / (target_years * 12)
        monthly_saving_ratio = (monthly_saving / (income * 10000 / 12)) if income > 0 else 0
        
        # 실현 가능성 평가
        feasibility = "높음" if monthly_saving_ratio <= 0.3 else "보통" if monthly_saving_ratio <= 0.5 else "낮음"
        
        plan = {
            'scenario': scenario,
            'target_amount': target_amount,
            'target_years': target_years,
            'monthly_saving': monthly_saving,
            'monthly_saving_ratio': monthly_saving_ratio,
            'feasibility': feasibility,
            'risk_match': self._assess_risk_match(scenario['risk_level'], risk_tolerance)
        }
        
        return plan
    
    def _assess_risk_match(self, scenario_risk, user_risk_tolerance):
        """시나리오 위험도와 사용자 위험성향 매칭"""
        risk_mapping = {
            'very_low': 1, 'low': 2, 'medium': 3, 'medium-high': 4, 'high': 5
        }
        
        scenario_score = risk_mapping.get(scenario_risk, 3)
        user_score = user_risk_tolerance
        
        diff = abs(scenario_score - user_score)
        
        if diff <= 1:
            return "매우 적합"
        elif diff <= 2:
            return "적합"
        else:
            return "위험성향 불일치"

In [11]:
# =====================================
# 3. 통합 추천 시스템
# =====================================

def integrated_recommendation(multilabel_model, user_features, selected_scenario=None):
    """포트폴리오 밸런싱 + 시나리오 기반 통합 추천"""
    
    print("\n🎯 **통합 맞춤 추천 분석**")
    print("=" * 50)
    
    # 기본 AI 예측
    user_input = user_features.reshape(1, -1)
    probabilities = multilabel_model.predict_proba(user_input)
    
    predictions = {}
    target_names = {
        'LIQ': '유동성자산',
        'CDS': '양도성예금증서', 
        'NMMF': '비머니마켓펀드',
        'STOCKS': '주식보유',
        'RETQLIQ': '퇴직준비금유동성'
    }
    
    for i, (label, name) in enumerate(target_names.items()):
        predictions[label] = probabilities[i][0][1]
    
    # 1. 포트폴리오 분석
    portfolio_balancer = PortfolioBalancer()
    portfolio_analysis = portfolio_balancer.analyze_current_portfolio(predictions, user_features[4])
    rebalancing_advice = portfolio_balancer.get_rebalancing_advice(portfolio_analysis, user_features)
    
    print("".join(rebalancing_advice['advice']))
    
    # 2. 시나리오 추천
    scenario_system = ScenarioRecommendationSystem()
    
    if selected_scenario:
        # 특정 시나리오 분석
        plan = scenario_system.create_scenario_plan(selected_scenario, user_features)
        if plan:
            print(f"\n🎯 **{plan['scenario']['name']} 시나리오 분석**")
            print(f"목표: {plan['scenario']['description']}")
            print(f"필요 자금: {plan['target_amount']:,.0f}만원")
            print(f"목표 기간: {plan['target_years']}년")
            print(f"월 저축액: {plan['monthly_saving']:,.0f}만원 (소득 대비 {plan['monthly_saving_ratio']:.1%})")
            print(f"실현 가능성: {plan['feasibility']}")
            print(f"위험성향 매칭: {plan['risk_match']}")
    else:
        # 시나리오 추천
        recommended_scenarios = scenario_system.recommend_scenario(user_features)
        print(f"\n🎯 **추천 시나리오 TOP 3**")
        for i, (scenario_id, score) in enumerate(recommended_scenarios, 1):
            scenario = scenario_system.scenarios[scenario_id]
            print(f"{i}. {scenario['name']} (적합도: {score}점)")
            print(f"   {scenario['description']} | 기간: {scenario['typical_period']}")
    
    # 3. 통합 상품 추천
    print(f"\n💼 **맞춤 상품 추천**")
    print("=" * 30)
    
    # AI 예측 결과 표시
    print("🤖 AI 기본 예측:")
    for label, name in target_names.items():
        prob = predictions[label]
        status = "★ 추천" if prob >= 0.3 else "※ 검토" if prob >= 0.1 else "- 비추천"
        print(f"  {name}: {prob:.1%} {status}")
    
    # 포트폴리오 조정 반영
    print(f"\n⚖️ 포트폴리오 균형 조정:")
    print(f"우선 추천: {rebalancing_advice['recommendation']}")
    
    if rebalancing_advice['priority_assets'] == 'safe':
        print("  → 안전자산(예금/적금) 우선 추천")
        adjusted_recommendations = ['LIQ', 'CDS', 'RETQLIQ']
    else:
        print("  → 성장자산(주식/펀드) 우선 추천")
        adjusted_recommendations = ['STOCKS', 'NMMF']
    
    return {
        'ai_predictions': predictions,
        'portfolio_analysis': portfolio_analysis,
        'rebalancing_advice': rebalancing_advice,
        'recommended_scenarios': recommended_scenarios if not selected_scenario else None,
        'scenario_plan': plan if selected_scenario else None,
        'final_recommendations': adjusted_recommendations
    }

In [12]:
# =====================================
# 4. 대화형 시나리오 선택
# =====================================

def interactive_scenario_selection(user_features):
    """사용자와 대화형으로 시나리오 선택"""
    
    scenario_system = ScenarioRecommendationSystem()
    
    print("\n🎯 **생애 이벤트 기반 맞춤 추천**")
    print("=" * 50)
    print("현재 준비하고 계신 것이 있나요?")
    print()
    
    scenarios_list = list(scenario_system.scenarios.items())
    for i, (scenario_id, scenario) in enumerate(scenarios_list, 1):
        print(f"{i}. {scenario['name']}")
        print(f"   {scenario['description']} (기간: {scenario['typical_period']})")
    
    print(f"{len(scenarios_list) + 1}. 📊 일반적인 포트폴리오 추천 (시나리오 선택 안함)")
    print()
    
    try:
        choice = int(input("선택해주세요 (번호 입력): "))
        
        if 1 <= choice <= len(scenarios_list):
            selected_scenario = scenarios_list[choice - 1][0]
            print(f"\n✅ '{scenarios_list[choice - 1][1]['name']}' 시나리오를 선택하셨습니다.")
            return selected_scenario
        else:
            print(f"\n✅ 일반적인 포트폴리오 추천을 선택하셨습니다.")
            return None
            
    except (ValueError, IndexError):
        print(f"\n※ 잘못된 입력입니다. 일반 추천으로 진행합니다.")
        return None

In [13]:
# =====================================
# 5. 메인 실행 함수
# =====================================

def run_advanced_recommendation_system(multilabel_model, user_features):
    """고급 추천 시스템 실행"""
    
    print("🚀 **고급 AI 금융상품 추천 시스템**")
    print("=" * 60)
    
    # 사용자 프로필 표시
    age = user_features[4]
    income = user_features[5]
    risk_tolerance = user_features[6]
    marriage_status = user_features[12]
    children_count = user_features[13]
    assets = user_features[15]
    
    print(f"\n👤 **사용자 프로필**")
    print(f"나이: {age}세 | 연소득: {income:,}만원 | 총자산: {assets:,}만원")
    print(f"결혼: {'기혼' if marriage_status else '미혼'} | 자녀: {children_count}명")
    
    risk_level = ['매우안전형', '안전형', '중립형', '적극형', '공격형'][min(risk_tolerance-1, 4)]
    print(f"투자성향: {risk_level}")
    
    # 시나리오 선택
    selected_scenario = interactive_scenario_selection(user_features)
    
    # 통합 추천 실행
    results = integrated_recommendation(multilabel_model, user_features, selected_scenario)
    
    print(f"\n🎉 **추천 완료!**")
    print("=" * 30)
    print("위 분석을 바탕으로 실제 금융상품을 추천해드립니다.")
    
    return results

In [24]:
# =====================================
# 사용 예시
# =====================================

if __name__ == "__main__":
    # 예시 사용자 데이터 (실제 사용 시 get_user_input() 함수 결과 사용)
    sample_user = np.array([
        2,      # 연령대분류 (30대)
        2,      # 교육수준분류 (대졸)
        0,      # 사업농업소득
        500,    # 자본이득소득
        32,     # 연령
        3,      # 금융위험감수 (보통)
        1,      # 저축여부 (함)
        5500,   # 급여소득
        3,      # 금융위험회피
        2,      # 교육수준
        1,      # 성별 (남성)
        1,      # 결혼상태 (기혼)
        1,      # 자녀수
        2       # 직업분류 (전문가)
    ])
    
    print("📝 **시스템 구성 요소**")
    print("1. ✅ 포트폴리오 밸런싱: 연령별 최적 자산 배분 분석")
    print("2. ✅ 시나리오 기반 추천: 생애 이벤트별 맞춤 상품 추천")
    print("3. ✅ 통합 AI 분석: XGBoost + 포트폴리오 + 시나리오 결합")
    print("4. ✅ 대화형 인터페이스: 사용자 맞춤 시나리오 선택")

📝 **시스템 구성 요소**
1. ✅ 포트폴리오 밸런싱: 연령별 최적 자산 배분 분석
2. ✅ 시나리오 기반 추천: 생애 이벤트별 맞춤 상품 추천
3. ✅ 통합 AI 분석: XGBoost + 포트폴리오 + 시나리오 결합
4. ✅ 대화형 인터페이스: 사용자 맞춤 시나리오 선택


In [26]:
# 더 현실적인 사용자 입력 함수
def get_realistic_user_input():
    print("🎯 현실적인 특성 기반 입력")
    print("=" * 40)
    
    try:
        # 기본 입력
        education = int(input("● 교육수준 (1:고졸, 2:대졸, 3:대학원): "))
        saving_status = int(input("● 저축여부 (1:예, 0:아니오): "))
        gender = int(input("● 성별 (1:남성, 0:여성): "))
        marriage = int(input("● 결혼상태 (1:기혼, 0:미혼): "))
        children = int(input("● 자녀수: "))
        occupation = int(input("● 직업 (1:관리자, 2:전문가, 3:사무직, 4:서비스, 5:기타): "))
        
        # 더 현실적인 추정값들
        if marriage == 0:
            age = 28
            salary = 3500  # 현실적인 초봉
        elif children == 0:
            age = 32
            salary = 4500
        else:
            age = 35 + children * 2
            salary = 5000 + children * 500  # 자녀수에 따른 소득 증가
        
        # 교육수준별 소득 조정
        education_multiplier = [0, 0.8, 1.0, 1.2][education]
        salary = int(salary * education_multiplier)
        
        # 연령대
        age_group = min(5, max(1, (age - 20) // 10 + 1))
        
        # 위험성향 (더 보수적으로)
        if age < 30 and education >= 2:
            risk_tolerance = 3  # 보통 (4에서 낮춤)
        elif age < 40:
            risk_tolerance = 3  # 보통
        else:
            risk_tolerance = 2  # 안전형
            
        risk_aversion = 6 - risk_tolerance
        
        # 14개 특성 (더 균형있게)
        user_features = np.array([
            age_group,          # 연령대분류
            education,          # 교육수준분류
            0,                  # 사업농업소득
            0,                  # 자본이득소득
            age,               # 연령
            risk_tolerance,     # 금융위험감수
            saving_status,      # 저축여부
            salary,            # 급여소득 (현실적)
            risk_aversion,      # 금융위험회피
            education,          # 교육수준
            gender,            # 성별
            marriage,          # 결혼상태
            children,          # 자녀수
            occupation         # 직업
        ])
        
        print(f"\n📋 생성된 프로필:")
        print(f"나이: {age}세, 소득: {salary:,}만원, 위험성향: {risk_tolerance}")
        
        return user_features
        
    except Exception as e:
        print(f"❌ 오류: {e}")
        return None

# 새로운 입력으로 테스트
print("🔄 더 현실적인 입력으로 재테스트:")
user_data_realistic = get_realistic_user_input()

if user_data_realistic is not None:
    try:
        probabilities = multilabel_xgb.predict_proba(user_data_realistic.reshape(1, -1))
        
        print(f"\n🎯 수정된 예측 결과:")
        target_names = ['유동성자산', '양도성예금증서', '비머니마켓펀드', '주식보유', '퇴직준비금유동성']
        for i, name in enumerate(target_names):
            prob = probabilities[i][0][1]
            status = "★ 추천" if prob >= 0.3 else "※ 검토" if prob >= 0.1 else "- 비추천"
            print(f"  {name}: {prob:.1%} {status}")
            
    except Exception as e:
        print(f"❌ 예측 오류: {e}")

🔄 더 현실적인 입력으로 재테스트:
🎯 현실적인 특성 기반 입력


● 교육수준 (1:고졸, 2:대졸, 3:대학원):  2
● 저축여부 (1:예, 0:아니오):  1
● 성별 (1:남성, 0:여성):  0
● 결혼상태 (1:기혼, 0:미혼):  0
● 자녀수:  0
● 직업 (1:관리자, 2:전문가, 3:사무직, 4:서비스, 5:기타):  2



📋 생성된 프로필:
나이: 28세, 소득: 3,500만원, 위험성향: 3

🎯 수정된 예측 결과:
  유동성자산: 99.7% ★ 추천
  양도성예금증서: 0.0% - 비추천
  비머니마켓펀드: 0.0% - 비추천
  주식보유: 0.0% - 비추천
  퇴직준비금유동성: 0.1% - 비추천


In [29]:
# 현실적인 해석을 위한 조정된 추천 함수
def realistic_interpretation(probabilities, user_features):
    """불균형 데이터를 고려한 현실적 해석"""
    
    print("🎯 현실적 해석 결과:")
    print("=" * 40)
    
    target_names = ['유동성자산', '양도성예금증서', '비머니마켓펀드', '주식보유', '퇴직준비금유동성']
    
    # 실제 데이터 보유율 (기준점)
    baseline_rates = [0.985, 0.077, 0.201, 0.291, 0.591]
    
    for i, (name, baseline) in enumerate(zip(target_names, baseline_rates)):
        predicted_prob = probabilities[i][0][1]
        
        # 기준 대비 상대적 해석
        if predicted_prob >= baseline * 0.95:  # 기준의 95% 이상
            if baseline > 0.8:  # 이미 높은 보유율
                recommendation = "✅ 필수 상품"
                confidence = "높음"
            else:
                recommendation = "⭐ 강력 추천"
                confidence = "높음"
        elif predicted_prob >= baseline * 0.7:  # 기준의 70% 이상
            recommendation = "💡 추천"
            confidence = "보통"
        elif predicted_prob >= baseline * 0.5:  # 기준의 50% 이상
            recommendation = "※ 검토"
            confidence = "낮음"
        else:
            recommendation = "❌ 비추천"
            confidence = "매우 낮음"
        
        # 상대적 점수 계산 (0-100)
        relative_score = min(100, (predicted_prob / baseline) * 50)
        
        print(f"{name}:")
        print(f"  예측 확률: {predicted_prob:.1%}")
        print(f"  일반 보유율: {baseline:.1%}")
        print(f"  상대 점수: {relative_score:.0f}/100")
        print(f"  결론: {recommendation} (신뢰도: {confidence})")
        print()
    
    # 전체 추천 요약
    age = user_features[4]
    risk_tolerance = user_features[5]
    
    print("📋 종합 추천 요약:")
    print(f"👤 프로필: {age}세, 위험성향 {risk_tolerance}/5")
    
    if age < 30:
        print("🌟 청년층 맞춤 전략:")
        print("  - 유동성자산: 비상자금 기본 (3-6개월 생활비)")
        print("  - 적금/투자: 목돈 마련 및 자산 증식 병행")
    elif age < 50:
        print("💼 중년층 맞춤 전략:")
        print("  - 유동성자산: 안정성 확보")
        print("  - 다양화: 예금, 적금, 투자 균형")
    else:
        print("🏖️ 시니어층 맞춤 전략:")
        print("  - 유동성자산: 안전 최우선")
        print("  - 보수적 투자: 연금, 안전자산 중심")

# 현실적 해석 실행
realistic_interpretation(probabilities, user_data_realistic)

🎯 현실적 해석 결과:
유동성자산:
  예측 확률: 99.7%
  일반 보유율: 98.5%
  상대 점수: 51/100
  결론: ✅ 필수 상품 (신뢰도: 높음)

양도성예금증서:
  예측 확률: 0.0%
  일반 보유율: 7.7%
  상대 점수: 0/100
  결론: ❌ 비추천 (신뢰도: 매우 낮음)

비머니마켓펀드:
  예측 확률: 0.0%
  일반 보유율: 20.1%
  상대 점수: 0/100
  결론: ❌ 비추천 (신뢰도: 매우 낮음)

주식보유:
  예측 확률: 0.0%
  일반 보유율: 29.1%
  상대 점수: 0/100
  결론: ❌ 비추천 (신뢰도: 매우 낮음)

퇴직준비금유동성:
  예측 확률: 0.1%
  일반 보유율: 59.1%
  상대 점수: 0/100
  결론: ❌ 비추천 (신뢰도: 매우 낮음)

📋 종합 추천 요약:
👤 프로필: 28세, 위험성향 3/5
🌟 청년층 맞춤 전략:
  - 유동성자산: 비상자금 기본 (3-6개월 생활비)
  - 적금/투자: 목돈 마련 및 자산 증식 병행


In [30]:
# 상대적 중요도 기반 추천
def relative_importance_recommendation(probabilities):
    """확률의 상대적 순위로 추천"""
    
    print("\n🏆 상대적 중요도 순위:")
    print("=" * 30)
    
    target_names = ['유동성자산', '양도성예금증서', '비머니마켓펀드', '주식보유', '퇴직준비금유동성']
    
    # 확률 추출
    probs = [probabilities[i][0][1] for i in range(len(target_names))]
    
    # (이름, 확률) 쌍으로 정렬
    ranked_products = [(name, prob) for name, prob in zip(target_names, probs)]
    ranked_products.sort(key=lambda x: x[1], reverse=True)
    
    print("우선순위별 추천:")
    for i, (name, prob) in enumerate(ranked_products, 1):
        if i <= 2:
            priority = "🥇 최우선" if i == 1 else "🥈 2순위"
        elif i <= 4:
            priority = "🥉 고려대상"
        else:
            priority = "📋 참고"
            
        print(f"{i}. {name}: {prob:.1%} - {priority}")
    
    # 실용적 조합 추천
    print(f"\n💼 실용적 상품 조합:")
    if ranked_products[0][0] == '유동성자산':
        print("✅ 기본 조합: 예금 + 적금")
        print("✅ 확장 조합: 예금 + 적금 + 안전투자상품")
    else:
        print("✅ 맞춤 조합: 상위 2-3개 상품 조합")

# 상대적 중요도 실행
relative_importance_recommendation(probabilities)


🏆 상대적 중요도 순위:
우선순위별 추천:
1. 유동성자산: 99.7% - 🥇 최우선
2. 퇴직준비금유동성: 0.1% - 🥈 2순위
3. 주식보유: 0.0% - 🥉 고려대상
4. 양도성예금증서: 0.0% - 🥉 고려대상
5. 비머니마켓펀드: 0.0% - 📋 참고

💼 실용적 상품 조합:
✅ 기본 조합: 예금 + 적금
✅ 확장 조합: 예금 + 적금 + 안전투자상품


In [31]:
# 개선된 종합 평가 시스템
def comprehensive_evaluation(probabilities, user_features):
    """종합적인 평가 및 실용적 추천"""
    
    print("\n🎯 XGBoost AI 종합 평가")
    print("=" * 50)
    
    age = user_features[4]
    risk_tolerance = user_features[5]
    saving_status = user_features[6]
    salary = user_features[7]
    marriage = user_features[11]
    children = user_features[12]
    
    # 생애주기 분석
    if age < 30:
        life_stage = "🌟 자산 형성기"
        focus = "목돈 마련 + 투자 시작"
    elif age < 45:
        life_stage = "💼 자산 증식기"  
        focus = "다양한 투자 + 안정성"
    else:
        life_stage = "🏖️ 자산 보전기"
        focus = "안전성 + 노후 준비"
    
    print(f"📊 생애주기: {life_stage}")
    print(f"🎯 투자 초점: {focus}")
    
    # 맞춤 추천
    print(f"\n💡 AI 맞춤 추천:")
    
    if saving_status == 1:  # 저축 중
        print("✅ 기존 저축 습관 우수 - 다양화 추천")
        if risk_tolerance >= 4:
            print("  → 적극적 투자상품 고려")
        else:
            print("  → 안전한 적금/펀드 고려")
    else:
        print("💡 저축 습관 시작 - 기본 상품부터")
        print("  → 정기예금/적금으로 기초 다지기")
    
    if marriage == 1 and children > 0:
        print("👨‍👩‍👧‍👦 가족 책임 고려 - 안정성 우선")
        print("  → 교육비/생활비 대비 필요")
    
    print(f"\n🎉 결론: 현재 상황에 맞는 점진적 포트폴리오 구성을 추천합니다!")

# 종합 평가 실행
comprehensive_evaluation(probabilities, user_data_realistic)


🎯 XGBoost AI 종합 평가
📊 생애주기: 🌟 자산 형성기
🎯 투자 초점: 목돈 마련 + 투자 시작

💡 AI 맞춤 추천:
✅ 기존 저축 습관 우수 - 다양화 추천
  → 안전한 적금/펀드 고려

🎉 결론: 현재 상황에 맞는 점진적 포트폴리오 구성을 추천합니다!
